# GGA 02 — Analysis (reads the pipeline's files)

Consumes `narratives.csv`, `X.npy`, `pk_order.csv` (and, if present,
`narratives_clustered.csv`) from the pipeline notebook's `HOME` folder.
**Does not use arcpy** — runs in the Pro notebook or any Python with
pandas / scikit-learn / scipy / statsmodels.

Contents: typologies (cluster → merge → validate) · mechanism x
collision/near-miss table · e-bike composition and adjusted severity ·
embedding twins · field twins (v2, v3 + ordinal). Executed reference results
are noted in each section so a re-run can be checked against them.

## 0 — Load the pipeline artifacts
Edit `HOME`. `CANADA`/`US` are only needed as a fallback if `narratives.csv`
is an old 8-column version (the current pipeline writes all columns).

In [ ]:
HOME   = r"E:\GGA\Embeddings\typology_full"        # the pipeline notebook's HOME
CANADA = r"E:\GGA\data\Canada_incidents.xlsx"      # fallback only (old narratives.csv)
US     = r"E:\GGA\data\United_States_incidents.xlsx"

import os
import numpy as np
import pandas as pd

SEED = 1208
rng_master = np.random.default_rng(SEED)

narr = pd.read_csv(os.path.join(HOME, "narratives.csv"))
X = np.load(os.path.join(HOME, "X.npy"))
pk_order = pd.read_csv(os.path.join(HOME, "pk_order.csv"))["pk"].to_numpy()
narr_x = narr.set_index("pk").loc[pk_order].reset_index()
assert len(narr_x) == X.shape[0], "narratives.csv and X.npy are out of sync — re-run the pipeline"
print(f"corpus {len(narr_x)} narratives | X {X.shape} | "
      f"ebike {(narr_x.ebike=='Yes').mean():.1%} | injured {narr_x.injured.mean():.1%}")

## 1 — Enrich and normalize
Merges any columns missing from an old `narratives.csv` (by `pk`, from the raw
pulls), derives `year` / `age_at` (the raw `age` column is BIRTH YEAR), the
ordinal injury level, and the normalized categorical fields used by the
field-matching. Idempotent.

In [ ]:
NEED = ["injury", "p_type", "date", "age", "gender", "incident_with",
        "intersection", "turning", "terrain", "road_conditions", "sightlines",
        "cars_on_roadside", "helmet", "trip_purpose", "ebike_class"]
todo = [c for c in NEED if c not in narr_x.columns]
if todo:
    print("merging from raw pulls:", todo)
    ef = []
    for path in [CANADA, US]:
        t = pd.read_excel(path)
        ef.append(t[["pk"] + [c for c in todo if c in t.columns]])
    narr_x = narr_x.merge(pd.concat(ef).drop_duplicates("pk"), on="pk", how="left")

narr_x["year"] = pd.to_datetime(narr_x["date"], errors="coerce").dt.year
narr_x["age_at"] = narr_x["year"] - pd.to_numeric(narr_x["age"], errors="coerce")
narr_x.loc[(narr_x["age_at"] < 5) | (narr_x["age_at"] > 100), "age_at"] = np.nan

LEVEL = {"No injury": 0, "Injury, no treatment": 1, "Injury, saw family doctor": 2,
         "Injury, hospital emergency visit": 2, "Injury, hospitalized": 2}
narr_x["inj_level"] = narr_x["injury"].map(LEVEL)

DK = {"i don't know", "don't remember", "i don't remember", "unknown", "nan", "none", ""}
def cat(v):
    s = str(v).strip()
    return "unknown" if s.lower() in DK else s
def gender_lump(v):
    s = str(v)
    if s in ("[]", "nan", "None"): return "unknown"
    if "F" in s: return "F"
    if "M" in s: return "M"
    return "other"

narr_x["iw_full"] = narr_x["incident_with"].apply(cat)
narr_x["gender_l"] = narr_x["gender"].apply(gender_lump)
SOFT = ["intersection", "turning", "terrain", "road_conditions",
        "cars_on_roadside", "sightlines", "helmet", "trip_purpose"]
for c in SOFT:
    narr_x[c + "_m"] = narr_x[c].apply(cat)
SOFTM = [c + "_m" for c in SOFT] + ["gender_l"]
print(narr_x[["year", "age_at", "inj_level", "p_type"]].notna().mean().round(3).to_string())

## 2 — Typologies: cluster, merge, label

Finding from the executed run: the embedding space is a **continuum** (HDBSCAN
returns 100% noise at n=4,397), so we over-segment with KMeans k=14 and merge
by hand into 7 mechanisms. Loads `narratives_clustered.csv` if the pipeline/
earlier session already saved k=14 labels; otherwise recomputes (seconds,
deterministic under this seed and sklearn version). **If you recompute under a
different sklearn version, re-read the per-cluster term printout and adjust
MERGE before trusting the typology labels.**

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import TfidfVectorizer

CLUST_CSV = os.path.join(HOME, "narratives_clustered.csv")
loaded = False
if os.path.exists(CLUST_CSV):
    prev = pd.read_csv(CLUST_CSV)
    if "cluster" in prev.columns and prev["cluster"].nunique() == 14:
        narr_x = narr_x.merge(prev[["pk", "cluster"]], on="pk", how="left")
        loaded = narr_x["cluster"].notna().all()
        print("loaded saved k=14 labels:", loaded)
if not loaded:
    from sklearn.cluster import KMeans
    Z = PCA(n_components=15, random_state=SEED).fit_transform(normalize(X))
    narr_x["cluster"] = KMeans(n_clusters=14, n_init=10, random_state=SEED).fit_predict(Z)
    print("computed KMeans k=14")

# read the clusters: top terms + samples (verify MERGE below still fits)
vec = TfidfVectorizer(stop_words="english", max_features=4000, ngram_range=(1, 2), min_df=3)
T = vec.fit_transform(narr_x["details_masked"])
terms = np.asarray(vec.get_feature_names_out())
lab = narr_x["cluster"].values
for c in sorted(set(lab)):
    in_c = lab == c
    score = T[in_c].mean(axis=0).A1 - T[~in_c].mean(axis=0).A1
    print(f"C{c:<2d} n={in_c.sum():<4d} inj={narr_x.loc[in_c,'injured'].mean():.0%} | "
          + ", ".join(terms[np.argsort(score)[::-1][:8]]))

# 14 -> 7 mechanism merge (derived by reading the 2026-10-19 run; verify above)
MERGE = {3: "contact_crash", 4: "contact_crash",
         6: "not_seen",
         2: "unsafe_pass", 10: "unsafe_pass", 13: "unsafe_pass",
         0: "failed_yield", 1: "failed_yield", 5: "failed_yield",
         8: "infrastructure", 12: "infrastructure",
         9: "harassment",
         7: "signal_violation", 11: "signal_violation"}
narr_x["typology"] = narr_x["cluster"].map(MERGE)
print("\n", narr_x["typology"].value_counts().to_string())

## 3 — Typology validation + mechanism x collision/near-miss

Reference (2026-10-19 run): injury gradient 11% (signal_violation) -> 32%
(contact_crash), chi2=129 p~2e-25; e-bike composition NULL (chi2=2.4, p=0.88,
share flat 15.5-19.0%).

In [ ]:
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportion_confint

rows = []
for t, g in narr_x.groupby("typology"):
    lo, hi = proportion_confint(g["injured"].sum(), len(g), method="wilson")
    rows.append({"typology": t, "n": len(g), "injury": g["injured"].mean(),
                 "lo": lo, "hi": hi, "ebike_share": (g["ebike"] == "Yes").mean()})
summ = pd.DataFrame(rows).sort_values("injury", ascending=False)
print(summ.to_string(index=False,
      formatters={c: "{:.1%}".format for c in ["injury", "lo", "hi", "ebike_share"]}))
print("injury x typology:", chi2_contingency(pd.crosstab(narr_x.typology, narr_x.injured))[:2])
print("ebike  x typology:", chi2_contingency(pd.crosstab(narr_x.typology, narr_x.ebike))[:2])
summ.to_csv(os.path.join(HOME, "typology_summary.csv"), index=False)

# mechanism x collision/near-miss (also the p_type validity check:
# contact_crash should be collision-dominated, signal/harassment near-miss-heavy)
d = narr_x.dropna(subset=["typology", "p_type"])
counts = pd.crosstab(d["typology"], d["p_type"], margins=True, margins_name="TOTAL")
print("\nCOUNTS\n", counts.to_string())
row_pct = pd.crosstab(d["typology"], d["p_type"], normalize="index")
print("\nROW % (within mechanism)\n", row_pct.applymap("{:.1%}".format).to_string())
out = counts.drop(index="TOTAL").copy()
for c in row_pct.columns:
    out[str(c) + "_pct"] = row_pct[c]
out["share_of_corpus"] = d["typology"].value_counts(normalize=True)
out.to_csv(os.path.join(HOME, "mechanism_ptype_table.csv"))
print("\nsaved ->", os.path.join(HOME, "mechanism_ptype_table.csv"))

## 4 — Adjusted e-bike severity (regression)

Reference: crude e-bike injury share is LOWER, but it is a **report-year
artifact** (injury share of all reports fell ~26% in 2017 to ~14% in 2025;
e-bike reports are recent). Adjusted OR = 0.90 (95% CI 0.69-1.18) — null.

In [ ]:
import statsmodels.api as sm

print("injury share by year:\n", narr_x.groupby("year")["injured"].agg(["mean", "count"]).round(3).to_string())
print("\np_type mix:\n", pd.crosstab(narr_x["ebike"], narr_x["p_type"], normalize="index").round(3).to_string())
col = narr_x[narr_x["p_type"] == "collision"]
print("\ninjury given collision:\n", col.groupby("ebike")["injured"].agg(["mean", "count"]).round(3).to_string())

m = narr_x.dropna(subset=["age_at", "typology", "year"]).copy()
Xd = pd.get_dummies(m["typology"], drop_first=True).astype(float)
Xd["age_at"] = m["age_at"]
Xd["year_c"] = m["year"] - 2021
Xd["ebike"] = (m["ebike"] == "Yes").astype(float)
fit = sm.Logit(m["injured"].values, sm.add_constant(Xd)).fit(disp=0)
or_, (lo, hi) = np.exp(fit.params["ebike"]), np.exp(fit.conf_int().loc["ebike"])
print(f"\nAdjusted e-bike injury OR (typology + age + year): {or_:.2f} ({lo:.2f}-{hi:.2f}), n={len(m)}")

## 5 — Embedding twins

Match each e-bike report to the most cosine-similar pedal narrative (same
country, year +/-1, caliper 0.60). Reference: with replacement -3.6pp p=0.048
(inflated by twin reuse — methods lesson); **without replacement 523 pairs,
-1.9pp, p=0.40 — null.**

In [ ]:
from sklearn.preprocessing import normalize as l2norm
from statsmodels.stats.contingency_tables import mcnemar

CALIPER = 0.60
Xn = l2norm(X)
ok = narr_x["year"].notna() & narr_x["injured"].notna()
eb_idx = np.where(ok & (narr_x["ebike"] == "Yes"))[0]
pd_idx = np.where(ok & (narr_x["ebike"] == "No"))[0]
pd_year = narr_x["year"].values[pd_idx]
pd_ctry = narr_x["country"].values[pd_idx]

pairs = []
for i in eb_idx:
    yi, ci = narr_x["year"].iat[i], narr_x["country"].iat[i]
    cand = pd_idx[(np.abs(pd_year - yi) <= 1) & (pd_ctry == ci)]
    if len(cand) == 0:
        continue
    sims = Xn[cand] @ Xn[i]
    if sims.max() >= CALIPER:
        pairs.append((i, cand[np.argmax(sims)], float(sims.max())))
sim = np.array([s for _, _, s in pairs])
print(f"matched within caliper: {len(pairs)}/{len(eb_idx)} "
      f"| similarity 25/50/75: {np.round(np.percentile(sim, [25, 50, 75]), 3)}")

def paired_mcnemar(idx_pairs, label):
    e = narr_x["injured"].values[[i for i, j in idx_pairs]].astype(int)
    t = narr_x["injured"].values[[j for i, j in idx_pairs]].astype(int)
    b = int(((e == 1) & (t == 0)).sum()); c = int(((e == 0) & (t == 1)).sum())
    res = mcnemar([[0, b], [c, 0]], exact=(b + c < 25))
    print(f"{label}: {len(idx_pairs)} pairs | ebike {e.mean():.1%} vs twins {t.mean():.1%}"
          f" | b={b} c={c} | p={res.pvalue:.4f}")

# without replacement (primary): best pairs first, each pedal twin used once
used, uniq = set(), []
for i, j, s in sorted(pairs, key=lambda t: -t[2]):
    if j not in used:
        used.add(j); uniq.append((i, j))
paired_mcnemar([(i, j) for i, j, _ in pairs], "EMBEDDING TWINS, with repl (anti-conservative)")
paired_mcnemar(uniq, "EMBEDDING TWINS, without repl (primary)")

## 6 — Field twins (v2 and v3) + ordinal severity

v2: hard match country + year+/-1 + full-level incident_with; soft agreement
score over 10 circumstance fields; without replacement, randomized tie-breaks.
Reference: -4.0pp p=0.042 — shown to carry **near-miss-mix leakage** (fields
can't see event resolution). v3 adds p_type to the hard match (estimand: same
circumstances AND same resolution). Ordinal ladder within v3 pairs — reference:
null everywhere (all pairs p=0.32; collision pairs 0.98 vs 1.04 p=0.63;
both-injured pairs 1.44 vs 1.51 p=0.44, n=82, limited power).

In [ ]:
from scipy.stats import binomtest, wilcoxon
rng = np.random.default_rng(SEED)

def agree_score(r, cand):
    s = np.zeros(len(cand))
    for c in SOFTM:
        s += (cand[c].values == r[c])
    ra, ca = r["age_at"], cand["age_at"].values
    if pd.isna(ra):
        s += pd.isna(ca)
    else:
        s += (~pd.isna(ca)) & (np.abs(ca - ra) <= 10)
    return s / 10.0

def field_match(hard_ptype):
    ok2 = ok & (narr_x["p_type"].notna() if hard_ptype else True)
    eb = np.where(ok2 & (narr_x["ebike"] == "Yes"))[0]
    P = narr_x[ok2 & (narr_x["ebike"] == "No")]
    keycols = ["country", "iw_full"] + (["p_type"] if hard_ptype else [])
    buckets = {k: g for k, g in P.groupby(keycols)}
    out = []
    for i in rng.permutation(eb):
        r = narr_x.iloc[i]
        key = tuple(r[c] for c in keycols)
        g = buckets.get(key)
        if g is not None:
            g = g[np.abs(g["year"] - r["year"]) <= 1]
        if g is None or len(g) == 0:
            continue
        sc = agree_score(r, g)
        j_lab = g.index[rng.choice(np.flatnonzero(sc == sc.max()))]
        out.append((i, narr_x.index.get_loc(j_lab)))
        buckets[key] = buckets[key].drop(j_lab)
    return out, len(eb)

v2, n_eb = field_match(hard_ptype=False)
print(f"FIELD TWINS v2: matched {len(v2)}/{n_eb}")
paired_mcnemar(v2, "  v2 binary (w/o repl)")

v3, n_eb3 = field_match(hard_ptype=True)
print(f"\nFIELD TWINS v3 (+p_type): matched {len(v3)}/{n_eb3}")
paired_mcnemar(v3, "  v3 binary (w/o repl)")

# ordinal within v3 pairs
ei3 = [i for i, _ in v3]; ti3 = [j for _, j in v3]
le, lt = narr_x["inj_level"].values[ei3], narr_x["inj_level"].values[ti3]
m3 = ~np.isnan(le) & ~np.isnan(lt)
le, lt = le[m3].astype(int), lt[m3].astype(int)
pt3 = narr_x["p_type"].values[ei3][m3]
print("\npaired 3x3 (rows=ebike, cols=twin):\n",
      pd.crosstab(pd.Series(le, name="ebike"), pd.Series(lt, name="twin")).to_string())

def ordinal_tests(a, b, label):
    worse = int((a > b).sum()); better = int((a < b).sum())
    if worse + better == 0:
        print(f"{label}: no discordant pairs"); return
    sp = binomtest(worse, worse + better).pvalue
    try:
        wp = wilcoxon(a, b, zero_method="wilcox").pvalue
    except ValueError:
        wp = float("nan")
    print(f"{label}: n={len(a)} | mean ebike {a.mean():.3f} vs twins {b.mean():.3f}"
          f" | worse {worse} vs {better} | sign p={sp:.4f} | Wilcoxon p={wp:.4f}")

ordinal_tests(le, lt, "ALL pairs")
for pt in ["collision", "nearmiss"]:
    s = pt3 == pt
    ordinal_tests(le[s], lt[s], f"  {pt}")
bi = (le >= 1) & (lt >= 1)
if bi.sum() >= 10:
    ordinal_tests(le[bi], lt[bi], "  both-injured (severity|injury)")
else:
    print(f"  both-injured: only {bi.sum()} pairs — descriptive only")

## Final finding (as of 2026-10-20)

In ~4,400 rider-reported incidents, e-bike and pedal riders experience the
same mix of incident mechanisms, and matched comparisons — on circumstances,
on event type, and on the narrative itself — show no difference in injury
occurrence or severity, including among pairs where both riders were injured.
The apparently lower injury share in e-bike reports is fully explained by
reporting era. Limits: no exposure denominator; outcomes conditional on the
decision to report; limited power in the severity-given-injury stratum.

Methods lessons demonstrated along the way: (1) report-year confounding,
(2) reused-control inflation in with-replacement matching, (3) outcome-mix
leakage in field-based matching.